# AIS Regional Filtering — PANYNJ Scope

## Purpose

This notebook filters the full January 2024 AIS dataset down to vessels relevant to the PANYNJ (Port Authority of New York and New Jersey) region.

The full US AIS dataset is too large to process in full on this machine (16GB RAM). Focusing on a single major port region keeps the dataset manageable and analytically meaningful.

## Approach

- validate schema consistency across all 31 daily files
- combine daily files into a single combined dataset
- identify all vessels that transmitted at least once within the PANYNJ bounding box
- retain the full movement scope (all transmissions) for those vessels
- write the scoped dataset to processed output

## Output

`AIS_2024_01_panynj_scope.csv`

One row = one AIS transmission. Dataset is not yet cleaned.

In [1]:
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

print("Imports loaded.")

Imports loaded.


In [2]:
def find_project_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(5):
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
        p = p.parent
    raise RuntimeError("Project root not found. Run Jupyter from project root or keep /data and /notebooks in place.")

In [3]:
NOTEBOOK_PATH = Path.cwd()

PROJECT_ROOT = find_project_root(NOTEBOOK_PATH)

RAW_DIR = PROJECT_ROOT / "Data" / "Raw"
PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# all the AIS is for the year 2024
RUN_TS = datetime.now(timezone.utc).isoformat(timespec="seconds")

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RUN_TS: {RUN_TS}")
print(f"RAW_DIR exists: {RAW_DIR.exists()}")
print(f"PROCESSED_DIR exists: {PROCESSED_DIR.exists()}")

AIS_files_paths = []
for day in range(1,32):
    if day < 10 : 
        file_name = f"AIS_2024_01_0{day}.csv"
    else:
        file_name = f"AIS_2024_01_{day}.csv"
    file_dir = RAW_DIR / file_name
    print(f"================================================\n{file_name} exists: {file_dir.exists()}")
    if file_dir.exists():
        print(f"{file_name} size (MB): {file_dir.stat().st_size/1e6:.1f}")
        AIS_files_paths.append(file_dir)

print("\nfiles Names:\n" + "\n".join(Path(p).name for p in AIS_files_paths))
print("\nPaths appended successfully")

PROJECT_ROOT: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Maritime Logistics Intelligence
RUN_TS: 2026-05-07T16:02:14+00:00
RAW_DIR exists: True
PROCESSED_DIR exists: True
AIS_2024_01_01.csv exists: True
AIS_2024_01_01.csv size (MB): 807.5
AIS_2024_01_02.csv exists: True
AIS_2024_01_02.csv size (MB): 808.2
AIS_2024_01_03.csv exists: True
AIS_2024_01_03.csv size (MB): 808.0
AIS_2024_01_04.csv exists: True
AIS_2024_01_04.csv size (MB): 784.9
AIS_2024_01_05.csv exists: True
AIS_2024_01_05.csv size (MB): 819.2
AIS_2024_01_06.csv exists: True
AIS_2024_01_06.csv size (MB): 808.1
AIS_2024_01_07.csv exists: True
AIS_2024_01_07.csv size (MB): 802.8
AIS_2024_01_08.csv exists: True
AIS_2024_01_08.csv size (MB): 799.2
AIS_2024_01_09.csv exists: True
AIS_2024_01_09.csv size (MB): 780.6
AIS_2024_01_10.csv exists: True
AIS_2024_01_10.csv size (MB): 778.5
AIS_2024_01_11.csv exists: True
AIS_2024_01_11.csv size (MB): 749.1
AIS_2024_01_12.csv exists: True
AIS_2024_01_12.csv size (MB)

In [4]:
#ref_df will be AIS_2024_01_01.csv 
ref_file = AIS_files_paths[0]
ref_df = pd.read_csv(ref_file, nrows=100)

ref_columns = list(ref_df.columns)
ref_dtypes = ref_df.dtypes
ref_col_count = len(ref_columns)

rows = []

for file in AIS_files_paths:
    df = pd.read_csv(file, nrows=100)

    cols = list(df.columns)
    dtypes = df.dtypes

    rows.append({
        "file": file.name,
        "column_names_match": set(cols) == set(ref_columns),
        "column_count_match": len(cols) == ref_col_count,
        "column_order_match": cols == ref_columns,
        "dtype_match": dtypes.equals(ref_dtypes)
    })

schema_check = pd.DataFrame(rows)

display(schema_check)

failures = schema_check[
    (~schema_check["column_names_match"]) |
    (~schema_check["column_count_match"]) |
    (~schema_check["column_order_match"]) |
    (~schema_check["dtype_match"])
]

print(f"\nFiles with schema mismatches: {len(failures)}")

if not failures.empty:
    display(failures)
else:
    print("\nAll files share identical schema.")

,file,column_names_match,column_count_match,column_order_match,dtype_match
0,AIS_2024_01_01.csv,True,True,True,True
1,AIS_2024_01_02.csv,True,True,True,True
2,AIS_2024_01_03.csv,True,True,True,False
3,AIS_2024_01_04.csv,True,True,True,False
4,AIS_2024_01_05.csv,True,True,True,True
5,AIS_2024_01_06.csv,True,True,True,True
6,AIS_2024_01_07.csv,True,True,True,True
7,AIS_2024_01_08.csv,True,True,True,True
8,AIS_2024_01_09.csv,True,True,True,True
9,AIS_2024_01_10.csv,True,True,True,True



Files with schema mismatches: 7


,file,column_names_match,column_count_match,column_order_match,dtype_match
2,AIS_2024_01_03.csv,True,True,True,False
3,AIS_2024_01_04.csv,True,True,True,False
10,AIS_2024_01_11.csv,True,True,True,False
12,AIS_2024_01_13.csv,True,True,True,False
13,AIS_2024_01_14.csv,True,True,True,False
19,AIS_2024_01_20.csv,True,True,True,False
24,AIS_2024_01_25.csv,True,True,True,False


In [5]:
dtype_mismatch_rows = []

for file in failures["file"]:
    file_path = RAW_DIR / file
    df = pd.read_csv(file_path, nrows=100)

    current_dtypes = df.dtypes

    for col in ref_columns:
        ref_dtype = str(ref_dtypes[col])
        cur_dtype = str(current_dtypes[col])

        if ref_dtype != cur_dtype:
            dtype_mismatch_rows.append({
                "file": file,
                "column": col,
                "reference_dtype": ref_dtype,
                "file_dtype": cur_dtype
            })

dtype_mismatches = pd.DataFrame(dtype_mismatch_rows)

display(dtype_mismatches)

print("\nMismatch counts by column:")
print(dtype_mismatches["column"].value_counts())

,file,column,reference_dtype,file_dtype
0,AIS_2024_01_03.csv,VesselType,int64,float64
1,AIS_2024_01_03.csv,Length,int64,float64
2,AIS_2024_01_03.csv,Width,int64,float64
3,AIS_2024_01_04.csv,Status,float64,int64
4,AIS_2024_01_04.csv,Cargo,float64,int64
5,AIS_2024_01_11.csv,Status,float64,int64
6,AIS_2024_01_11.csv,Cargo,float64,int64
7,AIS_2024_01_13.csv,Status,float64,int64
8,AIS_2024_01_13.csv,Cargo,float64,int64
9,AIS_2024_01_14.csv,Status,float64,int64



Mismatch counts by column:
column
Status        6
Cargo         6
VesselType    1
Length        1
Width         1
Name: count, dtype: int64


In [6]:
start_ts = datetime.now(timezone.utc)

print(f"Combine started: {start_ts.isoformat(timespec='seconds')}")

final_file = RAW_DIR / "AIS_2024_01_combined.csv"
temp_file = RAW_DIR / "AIS_2024_01_combined.tmp.csv"

expected_rows = 0
written_rows = 0
first_write = True

try:
    if temp_file.exists():
        temp_file.unlink()

    for file in AIS_files_paths:
        file_rows = 0

        for chunk in pd.read_csv(file, chunksize=500_000):
            chunk_rows = len(chunk)
            file_rows += chunk_rows
            expected_rows += chunk_rows

            chunk.to_csv(
                temp_file,
                mode="w" if first_write else "a",
                header=first_write,
                index=False
            )

            written_rows += chunk_rows
            first_write = False

        print(f"{file.name}: {file_rows:,} rows written")

    print(f"\nExpected rows: {expected_rows:,}")
    print(f"Written rows: {written_rows:,}")

    if written_rows != expected_rows:
        raise RuntimeError(
            f"Row count mismatch: expected {expected_rows:,}, wrote {written_rows:,}"
        )

    if final_file.exists():
        final_file.unlink()

    temp_file.rename(final_file)

    end_ts = datetime.now(timezone.utc)
    elapsed = end_ts - start_ts

    print("Row count verified")
    print(f"Combine finished: {end_ts.isoformat(timespec='seconds')}")
    print(f"Time taken: {elapsed}")
    print(f"Saved: {final_file}")

except Exception as e:
    if temp_file.exists():
        temp_file.unlink()

    print(f"Combine failed: {e}")

Combine started: 2026-05-07T16:02:15+00:00
AIS_2024_01_01.csv: 7,296,275 rows written
AIS_2024_01_02.csv: 7,295,616 rows written
AIS_2024_01_03.csv: 7,290,618 rows written
AIS_2024_01_04.csv: 7,085,305 rows written
AIS_2024_01_05.csv: 7,397,034 rows written
AIS_2024_01_06.csv: 7,287,838 rows written
AIS_2024_01_07.csv: 7,243,770 rows written
AIS_2024_01_08.csv: 7,210,061 rows written
AIS_2024_01_09.csv: 7,040,401 rows written
AIS_2024_01_10.csv: 7,024,515 rows written
AIS_2024_01_11.csv: 6,762,127 rows written
AIS_2024_01_12.csv: 7,008,571 rows written
AIS_2024_01_13.csv: 7,285,155 rows written
AIS_2024_01_14.csv: 5,746,483 rows written
AIS_2024_01_15.csv: 7,284,415 rows written
AIS_2024_01_16.csv: 7,182,595 rows written
AIS_2024_01_17.csv: 6,813,822 rows written
AIS_2024_01_18.csv: 6,972,898 rows written
AIS_2024_01_19.csv: 7,421,492 rows written
AIS_2024_01_20.csv: 7,420,105 rows written
AIS_2024_01_21.csv: 7,176,963 rows written
AIS_2024_01_22.csv: 6,961,234 rows written
AIS_2024_01

## PANYNJ Bounding Box

The region is defined by a bounding box covering the core port area and approach channels:

- **LAT:** 40.45 – 40.75
- **LON:** -74.25 – -73.85

This covers Upper and Lower New York Bay, Newark Bay, Kill Van Kull, Port Newark, Port Elizabeth, and the main approach from the Atlantic.

Coordinates align with standard AIS reception coverage for New York Harbor. A bounding box is sufficient for this scope — formal port polygon boundaries would require additional GIS data.

The filter works in two passes:
1. Identify all MMSIs that transmitted at least once inside the box.
2. Retain the full movement history for those vessels, not just in-region rows.

In [7]:
print("Selecting vessels linked to the PANYNJ region from combined AIS file...\n")

combined_file = RAW_DIR / "AIS_2024_01_combined.csv"
filtered_file = PROCESSED_DIR / "AIS_2024_01_panynj_scope.csv"
temp_file = PROCESSED_DIR / "AIS_2024_01_panynj_scope.tmp.csv"

lat_min, lat_max = 40.45, 40.75
lon_min, lon_max = -74.25, -73.85

selected_mmsi = set()
in_region_mmsi0_rows = 0

start_ts = datetime.now(timezone.utc)
print(f"Selection started: {start_ts.isoformat(timespec='seconds')}")

print("\nPass 1 — identifying vessels that entered the region...\n")

for chunk in pd.read_csv(combined_file, chunksize=500_000):

    in_region_mask = (
        chunk["LAT"].between(lat_min, lat_max) &
        chunk["LON"].between(lon_min, lon_max)
    )

    in_region_mmsi_chunk = (
        chunk.loc[
            in_region_mask,
            ["MMSI"]
        ]
    )

    valid_mmsi_chunk = (
        in_region_mmsi_chunk.loc[
            in_region_mmsi_chunk["MMSI"] != 0,
            "MMSI"
        ]
        .dropna()
        .astype("int64")
    )

    selected_mmsi.update(valid_mmsi_chunk.tolist())

    in_region_mmsi0_rows += (
        in_region_mmsi_chunk["MMSI"] == 0
    ).sum()

print("Pass 1 complete.")
print(f"Unique selected MMSI: {len(selected_mmsi):,}")
print(f"In-region MMSI=0 rows: {in_region_mmsi0_rows:,}")

if temp_file.exists():
    temp_file.unlink()

expected_rows = 0
written_rows = 0
first_write = True

print("\nPass 2 — writing scoped dataset...\n")

for chunk in pd.read_csv(combined_file, chunksize=500_000):

    in_region_mask = (
        chunk["LAT"].between(lat_min, lat_max) &
        chunk["LON"].between(lon_min, lon_max)
    )

    selected_mmsi_mask = (
        chunk["MMSI"].isin(selected_mmsi)
    )

    mmsi0_in_region_mask = (
        (chunk["MMSI"] == 0) &
        in_region_mask
    )

    keep_mask = (
        selected_mmsi_mask |
        mmsi0_in_region_mask
    )

    filtered_chunk = (
        chunk.loc[
            keep_mask
        ]
    )

    chunk_rows = len(filtered_chunk)
    expected_rows += chunk_rows

    if chunk_rows > 0:

        filtered_chunk.to_csv(
            temp_file,
            mode="w" if first_write else "a",
            header=first_write,
            index=False
        )

        written_rows += chunk_rows
        first_write = False

print("Pass 2 complete.")
print(f"Expected rows: {expected_rows:,}")
print(f"Written rows: {written_rows:,}")

if written_rows != expected_rows:
    if temp_file.exists():
        temp_file.unlink()
    raise RuntimeError(
        f"Row count mismatch: expected {expected_rows:,}, wrote {written_rows:,}"
    )

if filtered_file.exists():
    filtered_file.unlink()

temp_file.rename(filtered_file)

end_ts = datetime.now(timezone.utc)
elapsed = end_ts - start_ts

print("\nScoped dataset complete.")
print(f"Saved: {filtered_file}")
print(f"Finished: {end_ts.isoformat(timespec='seconds')}")
print(f"Time taken: {elapsed}")

Selecting vessels linked to the PANYNJ region from combined AIS file...

Selection started: 2026-05-07T17:11:07+00:00

Pass 1 — identifying vessels that entered the region...

Pass 1 complete.
Unique selected MMSI: 896
In-region MMSI=0 rows: 0

Pass 2 — writing scoped dataset...

Pass 2 complete.
Expected rows: 10,856,366
Written rows: 10,856,366

Scoped dataset complete.
Saved: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Maritime Logistics Intelligence\Data\Processed\AIS_2024_01_panynj_scope.csv
Finished: 2026-05-07T18:15:04+00:00
Time taken: 1:03:56.808277


In [8]:
# Final row count and vessel summary
scoped_df = pd.read_csv(filtered_file, usecols=["MMSI"])

print("Scoped dataset summary:")
print(f"  Total rows:     {len(scoped_df):,}")
print(f"  Unique vessels: {scoped_df['MMSI'].nunique():,}")
print(f"  File:           {filtered_file.name}")
print("\nStep 2 — Regional Filtering: Complete.")


Scoped dataset summary:
  Total rows:     10,856,366
  Unique vessels: 896
  File:           AIS_2024_01_panynj_scope.csv

Step 2 — Regional Filtering: Complete.
